<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Focused tests

**[C02-M05-L03 · Focused tests](https://learning.nextia-ai.com/courses/python/m05/focused-tests/)** · C02, Python for Practical AI Engineering · Module 5, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write tests with pytest for the normal case, the boundaries and the error paths, use fixtures for test data and temporary files, test the web client with a mock, and write a regression test that proves a bug stays fixed.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection: the setup installs httpx and pytest, and downloads the data files. No account. |
| **You should know** | `count_tickets.py` with logging and exit codes, from [Exceptions and logging](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/). `assert`, from [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/#check-each-function-alone). |
| **You do not need** | Your local project and its environment. The setup makes the same `ticket-cleaner` folder here, and installs pytest. |
| **Tested** | 2026-10-07, Python 3.14.6 with httpx 0.28.1 and pytest 9.1.1, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m05/focused-tests/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M05-L03-focused-tests/focused-tests-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `httpx` 0.28.1 and `pytest` 9.1.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `httpx 0.28.1`, `pytest 9.1.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check httpx==0.28.1 pytest==9.1.1
import httpx; print("httpx", httpx.__version__)
import pytest; print("pytest", pytest.__version__)

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, `tickets-excel.csv`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 15 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`), [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) (`files.py`, `split_records` and `build_summary`), [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) (`remote.py` and `try_mock.py`), [Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/) (`config.py` and `.env.example`), [Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/) (the two fixes in `parsing.py` and `files.py`), [Exceptions and logging](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/) (logging and exit codes).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
    "tickets-excel.csv": "cf0fbbce193bb1a595ce9207cc3f004aeedb118008e4ffa35585fda2f1167a95",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
import logging
from pathlib import Path

import httpx

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

OUTPUT_FILE = Path("reports/summary.json")

logger = logging.getLogger("ticket_cleaner")


def load_rows(settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if settings.source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(settings.source))


def main() -> int:
    settings = load_settings()
    logging.basicConfig(
        level=settings.log_level, format="%(levelname)s %(name)s: %(message)s"
    )
    logger.info("reading tickets from %s", settings.source)

    try:
        rows = load_rows(settings)
    except (OSError, ValueError, httpx.HTTPError) as error:
        logger.error("cannot read the tickets from %s: %s", settings.source, error)
        return 1

    tickets, rejected = split_records(rows)
    for record in rejected:
        logger.warning("row %d rejected: %s", record.row, "; ".join(record.problems))

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    logger.info("wrote %s", OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    if not isinstance(raw, dict):
        return ["not a record"]
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected
''',
    'ticket_cleaner/files.py': '''\
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8-sig", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\\n")
    os.replace(temporary, path)
''',
    'ticket_cleaner/remote.py': '''\
import httpx


def fetch_rows(
    url: str, token: str | None = None, client: httpx.Client | None = None
) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
''',
    'try_mock.py': '''\
import httpx

from ticket_cleaner.remote import fetch_rows


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def main() -> None:
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    print(fetch_rows("https://tickets.example/api/tickets", client=client))

    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    fetch_rows("https://tickets.example/api/tickets", client=client)


if __name__ == "__main__":
    main()
''',
    'ticket_cleaner/config.py': '''\
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)
    log_level: str


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
        log_level=os.environ.get("LOG_LEVEL", "WARNING").upper(),
    )
''',
    '.env.example': '''\
# Settings for ticket-cleaner. Set them as environment variables.
# TICKETS_SOURCE: a .csv or .json file, or "remote". Default: data/tickets.csv
TICKETS_SOURCE=data/tickets.csv
# TICKETS_URL: the web service for "remote". Default: the course's public file.
TICKETS_URL=
# TICKETS_API_TOKEN: secret. Never write a real token in this file.
TICKETS_API_TOKEN=
''',
    'data/broken.json': '''\
[
  {"id": "T-3001", "status": "open", "category": "billing", "priority": 2},
  null,
  {"id": "T-3003", "status": "closed", "category": "login", "priority": 3}
]
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

from pathlib import Path

REPORT_BUGS = [
    ("average_priority returns 0 for no tickets", "        return None\n", "        return 0\n"),
    ("average_priority does not round",
     "return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)",
     "return sum(ticket.priority for ticket in tickets) / len(tickets)"),
    ("count_by_category does not sort", "return dict(sorted(counts.items()))", "return counts"),
    ("build_summary counts all tickets as selected", '"selected": len(selected)', '"selected": len(tickets)'),
]

def check_report_tests():
    """Run tests/test_report.py, then break report.py in four ways: a test must fail each time."""
    result = run_quiet("-m", "pytest", "tests/test_report.py", "-q")
    last = result.stdout.strip().splitlines()[-1] if result.stdout.strip() else result.stderr.strip()
    if result.returncode != 0:
        print(f"Not yet: the tests do not all pass on the correct code: {last}")
        return
    passed = int(last.split(" passed")[0].split()[-1]) if " passed" in last else 0
    if passed < 4:
        print(f"Not yet: {passed} tests pass. Write at least 4.")
        return
    report = Path("ticket_cleaner/report.py")
    correct = report.read_text(encoding="utf-8")
    try:
        for description, good, bad in REPORT_BUGS:
            report.write_text(correct.replace(good, bad), encoding="utf-8")
            if run_quiet("-m", "pytest", "tests/test_report.py", "-q").returncode == 0:
                print(f"Not yet: your tests still pass when {description}. Add a test that finds it.")
                return
    finally:
        report.write_text(correct, encoding="utf-8")
    print(f"Check passed. ({passed} tests, and each of the four bugs makes a test fail.)")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. From assert to a test

A **test** is a function that checks one fact about your code with `assert`. A **test runner** finds all the tests, runs each one separately, and reports every failure. **pytest** looks for files `tests/test_*.py` and functions named `test_…`. A test passes if it ends without an exception. A good test name says the fact that it checks.

pytest is a dependency for development. The setup installed it. On your computer, add it to `requirements.txt` and install it, as in [Isolate dependencies](https://learning.nextia-ai.com/courses/python/m01/isolate-dependencies/#record-the-versions). Run the two cells: they update `requirements.txt` and make the `tests` folder.

In [ ]:
%%writefile requirements.txt
httpx==0.28.1
pytest==9.1.1

In [ ]:
from pathlib import Path

Path("tests").mkdir(exist_ok=True)

## 2. Worked example: the regression test first

A **regression** is a bug that comes back after it was fixed. A **regression test** reproduces the bug: it fails while the bug exists, and passes when it is fixed. Write it from the minimal failing example of [Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/#failure-2-a-wrong-result-with-no-error).

`tmp_path` is a **fixture**: a value that pytest makes and gives to a test that names it as a parameter. It is a new, empty folder for each test, so a test never changes the files of your project. Run the cell.

In [ ]:
%%writefile tests/test_files.py
import json

from ticket_cleaner.files import read_rows, write_json


def test_csv_with_byte_order_mark(tmp_path):
    path = tmp_path / "tickets.csv"
    path.write_text("id,status\nT-1001,open\n", encoding="utf-8-sig")
    assert read_rows(path) == [{"id": "T-1001", "status": "open"}]


def test_write_json_makes_the_folder(tmp_path):
    path = tmp_path / "reports" / "summary.json"
    write_json({"selected": 0}, path)
    assert json.loads(path.read_text(encoding="utf-8")) == {"selected": 0}
    assert not (tmp_path / "reports" / "summary.json.tmp").exists()

Run the tests. `python -m pytest` runs pytest with this Python, from the project folder, so the tests can import `ticket_cleaner`.

In [ ]:
run("-m", "pytest")

> **Check.** You should see `collected 2 items`, `tests/test_files.py ..` and `2 passed`. Each `.` is a passed test. (The `anyio` plugin comes with httpx.)

**Prove that the test can fail.** A test that passes with the bug and without it proves nothing. Run the cell: it puts the bug back (`utf-8` instead of `utf-8-sig` in `read_rows`), runs the tests, and fixes the file again.

In [ ]:
files = Path("ticket_cleaner/files.py")
fixed = files.read_text(encoding="utf-8")
files.write_text(fixed.replace('encoding="utf-8-sig"', 'encoding="utf-8"'), encoding="utf-8")
run("-m", "pytest")
files.write_text(fixed, encoding="utf-8")  # the fix again
print("files.py has the fix again.")

> **Check.** You should see `F.`, the failed test `test_csv_with_byte_order_mark`, and the lines with `E`: `At index 0 diff: {'\ufeffid': 'T-1001', 'status': 'open'} != {'id': 'T-1001', 'status': 'open'}`, then `1 failed, 1 passed`. The `>` marks the line that failed. That is the bug, found by a test in a fraction of a second.

Run the tests again, with the fix. You should see `2 passed`.

In [ ]:
run("-m", "pytest")

## 3. Test the contract: normal, boundary and error cases

For each function, test three kinds of input: **normal** (a usual input), **boundary** (the edges: priority `1` and `3` are valid, `0` and `4` are not, empty input) and **error path** (a missing field, `high`, `None` instead of a record). Bugs collect at boundaries.

Three new tools: `@pytest.fixture` makes your own fixture, `valid_raw`, which gives each test a **new** valid record. `@pytest.mark.parametrize` runs one test several times with different values. `del valid_raw["category"]` removes a key. Run the cell.

In [ ]:
%%writefile tests/test_parsing.py
import pytest

from ticket_cleaner.parsing import find_problems, split_records, to_ticket
from ticket_cleaner.records import Ticket


@pytest.fixture
def valid_raw():
    return {"id": "T-1001", "status": "open", "category": "billing", "priority": "2"}


def test_valid_record_has_no_problems(valid_raw):
    assert find_problems(valid_raw) == []


def test_text_is_cleaned(valid_raw):
    valid_raw["id"] = " t-1001 "
    valid_raw["status"] = "Open "
    expected = Ticket(id="T-1001", status="open", category="billing", priority=2)
    assert to_ticket(valid_raw) == expected


@pytest.mark.parametrize("priority", ["0", "4", "high", "2.5", "-1"])
def test_priority_outside_1_to_3_is_a_problem(valid_raw, priority):
    valid_raw["priority"] = priority
    assert find_problems(valid_raw) == [f"priority must be 1, 2 or 3, not {priority!r}"]


def test_missing_field_is_a_problem(valid_raw):
    del valid_raw["category"]
    assert find_problems(valid_raw) == ["missing category"]


def test_duplicate_id_is_rejected(valid_raw):
    tickets, rejected = split_records([valid_raw, dict(valid_raw)])
    assert len(tickets) == 1
    assert rejected[0].row == 2
    assert rejected[0].problems == ["duplicate id T-1001"]


def test_a_value_that_is_not_a_dictionary_is_rejected():
    tickets, rejected = split_records([None])
    assert tickets == []
    assert rejected[0].problems == ["not a record"]


def test_empty_input_gives_empty_results():
    assert split_records([]) == ([], [])

## 4. Test the web client without the network

The mock from [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/#test-without-the-network-mocked-responses) becomes two tests. `pytest.raises` checks the error path: the test passes only if the code in the `with` block raises that exception type. Run the cell, then run all the tests.

In [ ]:
%%writefile tests/test_remote.py
import httpx
import pytest

from ticket_cleaner.remote import fetch_rows

URL = "https://tickets.example/api/tickets"


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def test_fetch_rows_returns_the_records():
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    assert fetch_rows(URL, client=client) == [{"id": "T-2001", "status": "open"}]


def test_fetch_rows_raises_on_server_error():
    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    with pytest.raises(httpx.HTTPStatusError):
        fetch_rows(URL, client=client)

In [ ]:
run("-m", "pytest")

> **Check.** You should see `tests/test_files.py ..`, `tests/test_parsing.py ...........` (11 dots: 6 test functions, and the parametrized one counts 5 times), `tests/test_remote.py ..` and `15 passed`.

Run one group of tests by name with `-k`, and see each test's name with `-v`. Run the cell.

In [ ]:
run("-m", "pytest", "-k", "duplicate")
run("-m", "pytest", "-v", "tests/test_remote.py")

> **Check.** The first run shows `1 passed, 14 deselected`. The second shows the two test names of `test_remote.py`, each with `PASSED`.

## 5. Your turn: boundaries of the report

> **Your turn.** Complete `tests/test_report.py` in the next cell, with tests for `report.py`. Write at least these four, and give each a name that says the fact:
>
> 1. `average_priority` returns `None` for no tickets.
> 2. `average_priority` rounds to one decimal: priorities `1`, `1` and `2` give `1.3`.
> 3. `count_by_category` returns the categories in alphabetical order. (`list(a_dictionary)` gives its keys in order.)
> 4. `build_summary` for a status that no ticket has: `selected` is `0`, `by_category` is `{}`, and `average_priority` is `None`.
>
> The fixture `tickets` is ready. Run the cell, then the `run` cell. Then run the check cell: it breaks `report.py` in four ways, one at a time, and checks that one of your tests fails each time. It puts the correct `report.py` back at the end.

In [ ]:
%%writefile tests/test_report.py
import pytest

from ticket_cleaner.records import Ticket
from ticket_cleaner.report import average_priority, build_summary, count_by_category


@pytest.fixture
def tickets():
    return [
        Ticket(id="T-1", status="open", category="login", priority=1),
        Ticket(id="T-2", status="open", category="billing", priority=1),
        Ticket(id="T-3", status="closed", category="billing", priority=2),
    ]


def test_average_priority_of_no_tickets_is_none():
    assert average_priority([]) is None


def test_average_priority_is_rounded_to_one_decimal(tickets):
    assert average_priority(tickets) == 1.3


def test_categories_are_in_alphabetical_order(tickets):
    assert list(count_by_category(tickets)) == ["billing", "login"]


def test_summary_for_a_status_with_no_tickets(tickets):
    summary = build_summary(tickets, [], status="pending")
    assert summary["selected"] == 0
    assert summary["by_category"] == {}
    assert summary["average_priority"] is None

In [ ]:
run("-m", "pytest", "tests/test_report.py")

In [ ]:
check_report_tests()

## 6. Failure case: a test that passes for the wrong reason

Tomás writes a test that checks only that the list of problems is not empty. Run the cell: it saves the test, then breaks `find_problems` so that it reports the wrong problem for `high`, and runs the test.

In [ ]:
Path("tests/test_weak.py").write_text('''from ticket_cleaner.parsing import find_problems


def test_find_problems_for_bad_priority():
    raw = {"id": "T-1001", "status": "open", "category": "billing", "priority": "high"}
    problems = find_problems(raw)
    assert problems
''', encoding="utf-8")

parsing = Path("ticket_cleaner/parsing.py")
correct = parsing.read_text(encoding="utf-8")
parsing.write_text(correct.replace('f"priority must be 1, 2 or 3, not {priority!r}"', '"missing priority"'), encoding="utf-8")
run("-m", "pytest", "-q", "--tb=no", "tests/test_weak.py", "tests/test_parsing.py")

> **Check.** You should see `...FFFFF....`, five `FAILED tests/test_parsing.py::test_priority_outside_1_to_3_is_a_problem[…]` lines, and `5 failed, 7 passed`. (`--tb=no` leaves out the details of each failure.) The first dot is `test_weak.py`: it passes, although the function is broken. `assert problems` checks only that the list is not empty. The exact tests in `test_parsing.py` find the bug.

**Fix:** compare with the exact expected value, break the code on purpose to see that the test fails, and check that the count of tests goes up when you add one. Run the cell: it puts the correct `parsing.py` back, deletes the weak test, and runs all the tests.

In [ ]:
parsing.write_text(correct, encoding="utf-8")
Path("tests/test_weak.py").unlink()
run("-m", "pytest", "-q")

> **Check.** When your `test_report.py` is complete, you should see `19 passed` (or more, if you wrote more tests).

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/python/m05/focused-tests/). They count toward your certificate when you are signed in and enrolled. On your computer, install pytest in your project's environment, update the lock file, and keep the outputs of the failing and passing runs as evidence.

In this lesson you learned that pytest runs every `test_…` function in `tests/`, to test the normal case, the boundaries and the error paths with exact values, to use fixtures, `parametrize` and `pytest.raises`, to mock the network, and to check that each test fails without the fix.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m05/focused-tests/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.